In [8]:
# ============================================================
# SAFE OU-DFA with Regularized Horseshoe (NumPyro)
# ============================================================

import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

import jax
import jax.numpy as jnp
from jax import random

import numpyro
import numpyro.distributions as dist
from numpyro.infer import SVI, Trace_ELBO
from numpyro.optim import Adam

# ------------------------------------------------------------
# 1. Simulate irregular-time OU-DFA data
# ------------------------------------------------------------

def simulate_ou_dfa(
    key,
    N=50,       # subjects
    T=4,        # time points
    p=200,      # observed dim (scale later)
    K_true=5,
):
    keys = random.split(key, 6)

    # irregular times
    times = jnp.sort(
        random.uniform(keys[0], (N, T), minval=0.0, maxval=50.0),
        axis=1,
    )

    # true OU params
    rho = random.uniform(keys[1], (K_true,), minval=0.05, maxval=0.3)
    sigma_z = 0.5 * jnp.ones(K_true)

    # sparse loadings
    Lambda = random.normal(keys[2], (p, K_true)) * (
        random.bernoulli(keys[3], 0.1, (p, K_true))
    )

    sigma_y = 0.3 * jnp.ones(p)

    # latent states
    z = jnp.zeros((N, T, K_true))
    z = z.at[:, 0].set(random.normal(keys[4], (N, K_true)))

    for t in range(1, T):
        dt = times[:, t] - times[:, t - 1]
        phi = jnp.exp(-rho * dt[:, None])
        z = z.at[:, t].set(
            phi * z[:, t - 1] + sigma_z * random.normal(keys[5], (N, K_true))
        )

    # observations
    Y = jnp.einsum("ntk,pk->ntp", z, Lambda)
    Y = Y + sigma_y * random.normal(keys[0], Y.shape)

    mask = jnp.ones((N, T), dtype=bool)

    return Y, times, mask, z, Lambda


# ------------------------------------------------------------
# 2. SAFE OU-DFA with regularized horseshoe
# ------------------------------------------------------------

def ou_dfa_horseshoe_model(Y, times, mask, K, dt_min=1e-3):
    N, T, p = Y.shape

    # ---------- OU params ----------
    rho = numpyro.sample("rho", dist.LogNormal(0.0, 0.5).expand([K]))
    sigma_z = numpyro.sample("sigma_z", dist.LogNormal(-1.0, 0.5).expand([K]))

    # ---------- Horseshoe ----------
    tau = numpyro.sample("tau", dist.HalfCauchy(0.5))
    lambda_k = numpyro.sample("lambda_k", dist.HalfCauchy(1.0).expand([K]))
    c = numpyro.sample("c", dist.HalfNormal(1.0))

    # Regularized horseshoe (Piironen & Vehtari)
    lambda_tilde = (c * lambda_k) / jnp.sqrt(c**2 + (tau * lambda_k) ** 2)

    # Broadcast to (p, K)
    scale_Lambda = tau * lambda_tilde[None, :]   # shape (1, K) → (p, K)

    # ---------- Loadings ----------
    Lambda = numpyro.sample(
        "Lambda",
        dist.Normal(0.0, scale_Lambda).expand([p, K])
    )


    # ---------- Noise ----------
    sigma_y = numpyro.sample("sigma_y", dist.LogNormal(-1.0, 0.3).expand([p]))

    # ---------- Latent states ----------
    z_prev = numpyro.sample("z0", dist.Normal(0, 1).expand([N, K]))

    for t in range(T):
        if t > 0:
            dt = jnp.clip(times[:, t] - times[:, t - 1], dt_min)
            phi = jnp.exp(-rho * dt[:, None])
        else:
            phi = 0.0

        z_t = numpyro.sample(
            f"z_{t}", dist.Normal(phi * z_prev, sigma_z)
        )

        mean = jnp.einsum("nk,pk->np", z_t, Lambda)

        numpyro.sample(
            f"y_{t}",
            dist.Normal(mean, sigma_y),
            obs=jnp.where(mask[:, t, None], Y[:, t], jnp.nan),
        )

        z_prev = z_t


# ------------------------------------------------------------
# 3. Mean-field guide (SAFE)
# ------------------------------------------------------------

def ou_dfa_horseshoe_guide(Y, times, mask, K):
    N, T, p = Y.shape

    # OU
    rho_loc = numpyro.param("rho_loc", jnp.zeros(K))
    rho_scale = numpyro.param(
        "rho_scale", jnp.ones(K), constraint=dist.constraints.positive
    )
    numpyro.sample("rho", dist.LogNormal(rho_loc, rho_scale))

    sigma_z_loc = numpyro.param("sigma_z_loc", jnp.zeros(K))
    sigma_z_scale = numpyro.param(
        "sigma_z_scale", jnp.ones(K), constraint=dist.constraints.positive
    )
    numpyro.sample("sigma_z", dist.LogNormal(sigma_z_loc, sigma_z_scale))

    # Horseshoe
    tau_loc = numpyro.param("tau_loc", jnp.array(0.0))
    tau_scale = numpyro.param(
        "tau_scale", jnp.array(0.5), constraint=dist.constraints.positive
    )
    numpyro.sample("tau", dist.LogNormal(tau_loc, tau_scale))

    lambda_loc = numpyro.param("lambda_loc", jnp.zeros(K))
    lambda_scale = numpyro.param(
        "lambda_scale", jnp.ones(K), constraint=dist.constraints.positive
    )
    numpyro.sample("lambda_k", dist.LogNormal(lambda_loc, lambda_scale))

    c_loc = numpyro.param("c_loc", jnp.array(0.0))
    c_scale = numpyro.param(
        "c_scale", jnp.array(0.5), constraint=dist.constraints.positive
    )
    numpyro.sample("c", dist.LogNormal(c_loc, c_scale))

    # Loadings
    Lambda_loc = numpyro.param("Lambda_loc", jnp.zeros((p, K)))
    Lambda_scale = numpyro.param(
        "Lambda_scale",
        0.01 * jnp.ones((p, K)),
        constraint=dist.constraints.positive,
    )
    numpyro.sample("Lambda", dist.Normal(Lambda_loc, Lambda_scale))

    # Noise
    sigma_y_loc = numpyro.param("sigma_y_loc", jnp.zeros(p))
    sigma_y_scale = numpyro.param(
        "sigma_y_scale", jnp.ones(p), constraint=dist.constraints.positive
    )
    numpyro.sample("sigma_y", dist.LogNormal(sigma_y_loc, sigma_y_scale))

    # Latents
    z0_loc = numpyro.param("z0_loc", jnp.zeros((N, K)))
    z0_scale = numpyro.param(
        "z0_scale", jnp.ones((N, K)), constraint=dist.constraints.positive
    )
    numpyro.sample("z0", dist.Normal(z0_loc, z0_scale))

    for t in range(T):
        z_loc = numpyro.param(f"z_{t}_loc", jnp.zeros((N, K)))
        z_scale = numpyro.param(
            f"z_{t}_scale",
            jnp.ones((N, K)),
            constraint=dist.constraints.positive,
        )
        numpyro.sample(f"z_{t}", dist.Normal(z_loc, z_scale))


# ------------------------------------------------------------
# 4. Run everything
# ------------------------------------------------------------

def main():
    numpyro.set_platform("cpu")
    numpyro.set_host_device_count(1)

    key = random.PRNGKey(0)

    # ---- simulate ----
    Y, times, mask, z_true, Lambda_true = simulate_ou_dfa(
        key, N=40, T=4, p=200, K_true=5
    )

    # ---- fit ----
    K = 20  # overcomplete, horseshoe will prune

    svi = SVI(
        ou_dfa_horseshoe_model,
        ou_dfa_horseshoe_guide,
        Adam(5e-4),
        Trace_ELBO(),
    )

    state = svi.init(random.PRNGKey(1), Y, times, mask, K)

    for step in range(2000):
        state, loss = svi.update(state, Y, times, mask, K)
        if step % 200 == 0:
            print(f"step {step}, loss {loss:.2f}")

    params = svi.get_params(state)

    # ---- diagnostics ----
    lambda_mean = jnp.exp(params["lambda_loc"])
    print("Active factors:", jnp.sum(lambda_mean > 0.1))

    sparsity = jnp.mean(jnp.abs(params["Lambda_loc"]) < 1e-3)
    print("Loading sparsity:", sparsity)


if __name__ == "__main__":
    main()


step 0, loss 118314.14
step 200, loss 88988.54
step 400, loss 94043.61
step 600, loss 77219.54
step 800, loss 66593.62
step 1000, loss 257331.22
step 1200, loss 60155.57
step 1400, loss 153903.36
step 1600, loss 58217.20
step 1800, loss 67694.88
Active factors: 20
Loading sparsity: 0.15025


In [ ]:
import numpy as np
from numpy.linalg import inv, cholesky


In [ ]:
def ou_transition(rho, dt):
    A = np.exp(-rho * dt)
    return A


In [ ]:
def kalman_smoother(Y, times, Lambda, Q, Psi):
    """
    Y: (T, p)
    Lambda: (p, K)
    Q: (K, K)
    Psi: (p,)
    """
    T, p = Y.shape
    K = Lambda.shape[1]

    R = np.diag(Psi)
    H = Lambda

    # Storage
    m = np.zeros((T, K))
    P = np.zeros((T, K, K))
    m_pred = np.zeros_like(m)
    P_pred = np.zeros_like(P)

    # Init
    m[0] = 0
    P[0] = np.eye(K)

    # Forward pass
    for t in range(1, T):
        A = ou_transition(1.0, times[t] - times[t-1]) * np.eye(K)
        m_pred[t] = A @ m[t-1]
        P_pred[t] = A @ P[t-1] @ A.T + Q

        S = H @ P_pred[t] @ H.T + R
        Kt = P_pred[t] @ H.T @ inv(S)

        m[t] = m_pred[t] + Kt @ (Y[t] - H @ m_pred[t])
        P[t] = (np.eye(K) - Kt @ H) @ P_pred[t]

    # RTS smoother
    Ez = m.copy()
    Ezz = np.zeros_like(P)
    Ezz_lag = np.zeros_like(P)

    for t in reversed(range(T-1)):
        A = ou_transition(1.0, times[t+1] - times[t]) * np.eye(K)
        C = P[t] @ A.T @ inv(P_pred[t+1])
        Ez[t] += C @ (Ez[t+1] - m_pred[t+1])
        P[t] += C @ (P[t+1] - P_pred[t+1]) @ C.T

        Ezz_lag[t+1] = P[t+1] @ C.T + np.outer(Ez[t+1], Ez[t])

    for t in range(T):
        Ezz[t] = P[t] + np.outer(Ez[t], Ez[t])

    return Ez, Ezz, Ezz_lag


In [ ]:
def update_lambda_horseshoe(Ez_all, Y_all, tau, lam, Psi):
    """
    Ez_all: list of (T, K)
    Y_all: list of (T, p)
    """
    p = Y_all[0].shape[1]
    K = Ez_all[0].shape[1]

    Z = np.vstack(Ez_all)
    Y = np.vstack(Y_all)

    Lambda = np.zeros((p, K))

    for j in range(p):
        Sj = Z.T @ Z + np.diag(1 / (tau * lam[j])**2)
        bj = Z.T @ Y[:, j]
        Lambda[j] = np.linalg.solve(Sj, bj)

    return Lambda


In [ ]:
def update_ou_params(Ezz, Ezz_lag, dts):
    num = 0
    den = 0
    for t in range(1, len(dts)):
        num += np.trace(Ezz_lag[t])
        den += np.trace(Ezz[t-1])
    rho = -np.log(num / den) / np.mean(dts)
    return rho


In [ ]:
def update_noise(Y_all, Ez_all, Lambda):
    resid = []
    for Y, Ez in zip(Y_all, Ez_all):
        resid.append(Y - Ez @ Lambda.T)
    resid = np.vstack(resid)
    return np.var(resid, axis=0)


In [ ]:
def fit_ou_dfa(Y_all, times_all, K, max_iter=50):
    p = Y_all[0].shape[1]

    # Init
    Lambda = np.random.randn(p, K) * 0.01
    Q = np.eye(K)
    Psi = np.ones(p)
    tau = 1.0
    lam = np.ones((p, K))

    for it in range(max_iter):
        Ez_all, Ezz_all, Ezz_lag_all = [], [], []

        # E-step
        for Y, times in zip(Y_all, times_all):
            Ez, Ezz, Ezz_lag = kalman_smoother(Y, times, Lambda, Q, Psi)
            Ez_all.append(Ez)
            Ezz_all.append(Ezz)
            Ezz_lag_all.append(Ezz_lag)

        # M-step
        Lambda = update_lambda_horseshoe(Ez_all, Y_all, tau, lam, Psi)
        Psi = update_noise(Y_all, Ez_all, Lambda)

        print(f"Iter {it}: mean |Λ| = {np.mean(np.abs(Lambda)):.4f}")

    return Lambda, Psi


I. Quantitative Evaluation (reportable numbers)
1️⃣ Predictive log likelihood (held-out)
Strategy (recommended)

Hold out last visit per subject (natural for longitudinal data)

Fit model on earlier visits

Predict held-out

In [ ]:
import jax.numpy as jnp
import numpyro.distributions as dist

def predictive_loglik(Y_true, Y_pred_mean, sigma_y):
    ll = dist.Normal(Y_pred_mean, sigma_y).log_prob(Y_true)
    return ll.sum()


2️⃣ Reconstruction error (sanity check)

In [ ]:
def nrmse(Y, Y_hat):
    return jnp.linalg.norm(Y - Y_hat) / jnp.linalg.norm(Y)


3️⃣ Effective number of factors (horseshoe success)

In [ ]:
def effective_factors(Lambda_samples, threshold=0.05):
    Lambda_mean = jnp.mean(jnp.abs(Lambda_samples), axis=0)
    factor_strength = Lambda_mean.mean(axis=0)
    return jnp.sum(factor_strength > threshold), factor_strength


4️⃣ Temporal smoothness score (OU validation)

In [ ]:
def smoothness_score(Z):
    diffs = jnp.diff(Z, axis=1)
    return jnp.mean(diffs**2) / jnp.mean(Z**2)


5️⃣ Stability across random seeds (identifiability)

In [ ]:
from scipy.linalg import orthogonal_procrustes

def factor_similarity(Z1, Z2):
    R, _ = orthogonal_procrustes(Z1.reshape(-1, Z1.shape[-1]),
                                 Z2.reshape(-1, Z2.shape[-1]))
    return jnp.corrcoef(Z1 @ R, Z2)[0,1]


II. Qualitative Diagnostics (what reviewers actually look at)
6️⃣ Individual latent trajectories (MUST)

In [ ]:
import matplotlib.pyplot as plt

def plot_latent_trajectories(Z_mean, times, factor_id=0, n_subjects=20):
    plt.figure(figsize=(6,4))
    for i in range(n_subjects):
        plt.plot(times[i], Z_mean[i,:,factor_id], alpha=0.4)
    plt.xlabel("Age")
    plt.ylabel(f"Latent factor {factor_id}")
    plt.title("OU latent trajectories")
    plt.show()


7️⃣ Population mean ± uncertainty

In [ ]:
def plot_mean_trajectory(Z_samples, times, factor_id=0):
    Z_mean = Z_samples.mean(0)
    Z_std = Z_samples.std(0)

    tgrid = times.mean(0)
    plt.plot(tgrid, Z_mean.mean(0)[:,factor_id])
    plt.fill_between(
        tgrid,
        Z_mean.mean(0)[:,factor_id] - Z_std.mean(0)[:,factor_id],
        Z_mean.mean(0)[:,factor_id] + Z_std.mean(0)[:,factor_id],
        alpha=0.3
    )
    plt.title(f"Population OU factor {factor_id}")
    plt.show()


8️⃣ Factor correlation heatmap (degeneracy check)

In [ ]:
import seaborn as sns

def plot_factor_corr(Z_mean):
    Z_flat = Z_mean.reshape(-1, Z_mean.shape[-1])
    corr = jnp.corrcoef(Z_flat.T)
    sns.heatmap(corr, cmap="coolwarm", center=0)
    plt.title("Factor correlation")
    plt.show()


9️⃣ Loading matrix diagnostics (very important)
a) Heatmap of top factors

In [ ]:
def plot_loading_heatmap(Lambda_mean, top_k=10):
    order = jnp.argsort(jnp.max(jnp.abs(Lambda_mean), axis=1))
    sns.heatmap(Lambda_mean[order, :top_k], cmap="coolwarm", center=0)
    plt.title("Sparse loading structure")
    plt.show()


b) Sparsity histogram

In [ ]:
def plot_loading_hist(Lambda_mean):
    plt.hist(jnp.abs(Lambda_mean).flatten(), bins=100, log=True)
    plt.xlabel("|loading|")
    plt.ylabel("count")
    plt.title("Loading sparsity")
    plt.show()


c) Factor signatures (biological meaning)

In [ ]:
def top_features_per_factor(Lambda_mean, factor_id, feature_names, k=20):
    idx = jnp.argsort(jnp.abs(Lambda_mean[:,factor_id]))[::-1][:k]
    return [(feature_names[i], Lambda_mean[i,factor_id]) for i in idx]


10️⃣ Posterior predictive checks (gold standard)
a) Observed vs predicted

In [ ]:
def ppc_scatter(Y_true, Y_pred):
    plt.scatter(Y_true.flatten(), Y_pred.flatten(), alpha=0.1)
    plt.xlabel("Observed")
    plt.ylabel("Predicted")
    plt.title("Posterior predictive check")
    plt.show()


b) Time-resolved PPC

In [ ]:
def ppc_time(Y_true, Y_pred, times, subject_id=0, feature_id=0):
    plt.plot(times[subject_id], Y_true[subject_id,:,feature_id], 'o', label="obs")
    plt.plot(times[subject_id], Y_pred[subject_id,:,feature_id], '-', label="pred")
    plt.legend()
    plt.title("Time-resolved PPC")
    plt.show()


I. Additional Model Extensions (Methods-friendly)
1️⃣ Age-aligned latent trajectories (crucial with T=3–4)
Motivation

Subjects have:

sparse visits

different ages

overlapping age span (30–80)

We estimate population trajectories by binning age or continuous interpolation.

In [ ]:
def age_align(Z_samples, ages, age_grid):
    """
    Z_samples: (S, N, T, K)
    ages: (N, T)
    age_grid: (G,)
    """
    S, N, T, K = Z_samples.shape
    Z_aligned = jnp.zeros((S, N, len(age_grid), K))

    for g, a in enumerate(age_grid):
        w = jnp.exp(-0.5 * ((ages - a) / 2.0)**2)  # Gaussian kernel
        w = w / (w.sum(axis=1, keepdims=True) + 1e-8)
        Z_aligned = Z_aligned.at[:,:,g,:].set(
            jnp.einsum("sntk,nt->snk", Z_samples, w)
        )
    return Z_aligned


2️⃣ Disease progression / risk modeling (optional but strong)

If you have:

diagnosis

conversion indicator

survival time

In [ ]:
def risk_model(Z_mean, beta):
    logits = jnp.einsum("ntk,k->nt", Z_mean, beta)
    return logits


3️⃣ WAIC / PSIS-LOO (Bayesian rigor)
WAIC

In [ ]:
def waic(log_lik_samples):
    lppd = jnp.sum(jnp.log(jnp.mean(jnp.exp(log_lik_samples), axis=0)))
    p_waic = jnp.sum(jnp.var(log_lik_samples, axis=0))
    return -2 * (lppd - p_waic)


PSIS-LOO (recommended)

In [ ]:
import arviz as az
az.loo(idata)


II. Quantitative Evaluation (Methods-ready)
4️⃣ Factor recovery (synthetic validation)
Procrustes-aligned correlation

In [ ]:
from scipy.linalg import orthogonal_procrustes

def factor_recovery(Z_true, Z_est):
    R, _ = orthogonal_procrustes(
        Z_est.reshape(-1, Z_est.shape[-1]),
        Z_true.reshape(-1, Z_true.shape[-1])
    )
    Z_aligned = Z_est @ R
    return jnp.corrcoef(
        Z_aligned.reshape(-1),
        Z_true.reshape(-1)
    )[0,1]


5️⃣ Shrinkage effectiveness (horseshoe proof)

In [ ]:
def shrinkage_diagnostics(Lambda_samples):
    Lambda_abs = jnp.abs(Lambda_samples)
    active = (Lambda_abs > 0.05).mean(axis=(0,1))
    return active


6️⃣ Temporal coherence score (OU justification)

In [ ]:
def temporal_r2(Z, dt):
    dz = jnp.diff(Z, axis=1)
    return 1 - jnp.var(dz) / jnp.var(Z)


III. Visualization Suite (Reviewer-approved)
7️⃣ Population latent trajectories (key figure)

In [ ]:
def plot_population_trajectory(Z_aligned, age_grid, k=0):
    mean = Z_aligned.mean(axis=(0,1))
    std = Z_aligned.std(axis=(0,1))
    plt.plot(age_grid, mean[:,k])
    plt.fill_between(age_grid,
                     mean[:,k]-std[:,k],
                     mean[:,k]+std[:,k],
                     alpha=0.3)
    plt.xlabel("Age")
    plt.ylabel("Latent value")
    plt.title(f"Population OU factor {k}")
    plt.show()


8️⃣ Factor stability across runs (identifiability)

In [ ]:
def stability_matrix(Z_runs):
    R = len(Z_runs)
    K = Z_runs[0].shape[-1]
    S = jnp.zeros((R, R))
    for i in range(R):
        for j in range(R):
            S = S.at[i,j].set(factor_similarity(Z_runs[i], Z_runs[j]))
    return S


9️⃣ Loading structure visualization
a) Sorted heatmap
b) Feature importance bar plots
c) Sparsity vs factor index

These must be included for a methods paper.

🔟 Posterior predictive checks (mandatory)
Distributional PPC

In [ ]:
def ppc_hist(Y, Y_rep):
    plt.hist(Y.flatten(), bins=50, alpha=0.5, label="obs")
    plt.hist(Y_rep.flatten(), bins=50, alpha=0.5, label="rep")
    plt.legend()
    plt.show()
